# 07 — Series Comparativas: Evolución Temporal de Tecnologías Equivalentes

> **Fase 3 · Análisis de Tendencias — Framework Intelligence-to-Action (I2A)**

## Objetivo

Este cuaderno analiza la **evolución temporal** de tecnologías estrictamente equivalentes agrupadas en tres categorías competitivas:

| Batalla | Tecnologías | Pregunta clave |
|---|---|---|
| **Lenguajes** | Python · JavaScript · Java | ¿Cuándo Python superó a Java? |
| **Frameworks Web** | React · Angular · Vue | ¿Cuándo React dominó el ecosistema frontend? |
| **Bases de Datos** | PostgreSQL · MongoDB · MySQL | ¿SQL recuperó terreno frente a NoSQL? |

Se aplica un **suavizado estadístico** (Media Móvil de 3 meses) para aislar la tendencia estructural del ruido mensual. La serie resultante alimentará directamente el dashboard de la Fase 4.

---

In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import sys, os

# Asegurar que src/ sea importable
sys.path.append(os.path.abspath('..'))

# Rutas
DIM_Q   = '../data/datos_procesados/dim_questions.parquet'
FACT_T  = '../data/datos_procesados/fact_question_tags.parquet'
OUT_DIR = '../data/datos_procesados/eda'

## 0 · Carga y Unión de Datos (Lazy Evaluation)

Usamos `scan_parquet` para mantener la evaluación perezosa hasta el último momento.
La tabla de hechos (`fact_question_tags`) tiene la columna `Tags` (singular pregunta ↔ N etiquetas ya explotadas en Fase 2). La unimos con `dim_questions` para obtener la `CreationDate`.

Filtramos estrictamente al periodo **2015-2024** antes de materializar.

In [2]:
# Carga perezosa
lf_questions = pl.scan_parquet(DIM_Q)
lf_tags      = pl.scan_parquet(FACT_T)

# Inner join y filtrado temporal (2015-2024)
lf_base = (
    lf_tags
    .join(lf_questions.select('Id', 'CreationDate'), on='Id', how='inner')
    .filter(
        (pl.col('CreationDate').dt.year() >= 2015) &
        (pl.col('CreationDate').dt.year() <= 2024)
    )
    .rename({'Tags': 'Tag'})  # Normalizar nombre de columna
)

print('Plan de ejecucion (lazy):')
print(lf_base.explain())

Plan de ejecucion (lazy):
SELECT [col("Id"), col("Tags").alias("Tag"), col("CreationDate")]
  INNER JOIN:
  LEFT PLAN ON: [col("Id")]
    Parquet SCAN [../data/datos_procesados/fact_question_tags.parquet]
    PROJECT */2 COLUMNS
    ESTIMATED ROWS: 48050131
  RIGHT PLAN ON: [col("Id")]
    Parquet SCAN [../data/datos_procesados/dim_questions.parquet]
    PROJECT 2/6 COLUMNS
    SELECTION: [([(col("CreationDate").dt.year()) >= (2015)]) & ([(col("CreationDate").dt.year()) <= (2024)])]
    ESTIMATED ROWS: 16055694
  END INNER JOIN


### Función auxiliar: Serie mensual con Media Móvil

Definimos una función reutilizable que, dado un conjunto de tags y un LazyFrame base:
1. Filtra por los tags solicitados.
2. Agrupa por `Tag` + `YearMonth` con un conteo absoluto.
3. Calcula la **Media Móvil de 3 meses** usando la sintaxis vectorial pura de Polars:
   `pl.col('Count').rolling_mean(3).over('Tag')`
   *(Sin bucles, sin lambdas, sin `map_groups`.)*
4. Devuelve un DataFrame materializado listo para graficar.

In [3]:
def build_monthly_series(lf: pl.LazyFrame, tags: list) -> pl.DataFrame:
    """
    Construye la serie mensual suavizada para un conjunto de tags.
    
    Args:
        lf: LazyFrame base con columnas Tag, CreationDate.
        tags: Lista de tags a incluir.
    Returns:
        DataFrame con Tag, YearMonth (Date), Count, MA3.
    """
    df = (
        lf
        .filter(pl.col('Tag').is_in(tags))
        .with_columns(
            # Truncar la fecha al primer dia del mes para obtener granularidad mensual
            pl.col('CreationDate').dt.truncate('1mo').alias('YearMonth')
        )
        .group_by('Tag', 'YearMonth')
        .agg(pl.len().alias('Count'))
        .sort('Tag', 'YearMonth')
        .collect()
    )
    
    # Media Movil de 3 meses — vectorial, sin bucles
    df = df.with_columns(
        pl.col('Count')
          .rolling_mean(window_size=3)
          .over('Tag')
          .alias('MA3')
    )
    
    return df

In [4]:
def find_crossover(df: pl.DataFrame, tag_a: str, tag_b: str) -> dict:
    """
    Detecta el primer punto de cruce donde tag_a supera a tag_b en MA3.
    Retorna un dict con la fecha y los valores, o None si no hay cruce.
    """
    # Pivotar para comparar series lado a lado
    pivot = (
        df.filter(pl.col('Tag').is_in([tag_a, tag_b]))
        .pivot(on='Tag', index='YearMonth', values='MA3')
        .sort('YearMonth')
        .drop_nulls()
    )
    
    if tag_a not in pivot.columns or tag_b not in pivot.columns:
        return None
    
    pivot = pivot.with_columns(
        (pl.col(tag_a) > pl.col(tag_b)).alias('a_leads'),
    )
    
    # Detectar el primer mes donde a_leads cambia de False a True
    pivot = pivot.with_columns(
        pl.col('a_leads').shift(1).alias('prev_a_leads')
    )
    
    crossover = pivot.filter(
        pl.col('a_leads') & ~pl.col('prev_a_leads').fill_null(True)
    )
    
    if crossover.is_empty():
        return None
    
    row = crossover.row(0, named=True)
    return {
        'date': row['YearMonth'],
        'val_a': row[tag_a],
        'val_b': row[tag_b],
    }

---

## 1 · Batalla de Lenguajes: Python vs JavaScript vs Java

La hipótesis central del proyecto postula que **Python ha superado a Java como el segundo lenguaje más consultado en Stack Overflow**, impulsado por la explosión del ecosistema de ciencia de datos e inteligencia artificial. Aquí lo verificamos empíricamente.

In [5]:
LANGS = ['python', 'javascript', 'java']

df_langs = build_monthly_series(lf_base, LANGS)

# Detectar cruce Python > Java
cross_py_java = find_crossover(df_langs, 'python', 'java')
if cross_py_java:
    print(f"Cruce Python > Java detectado en: {cross_py_java['date']}")
else:
    print('No se detecto cruce Python > Java en el periodo analizado.')

Cruce Python > Java detectado en: 2017-06-01 00:00:00


In [6]:
# Grafico de lineas: Lenguajes
fig_langs = px.line(
    df_langs.to_pandas(),
    x='YearMonth', y='MA3', color='Tag',
    title='Evolucion Mensual — Python vs JavaScript vs Java (MA 3 meses)',
    labels={'MA3': 'Preguntas (Media Movil 3m)', 'YearMonth': '', 'Tag': 'Lenguaje'},
    color_discrete_map={'python': '#3776AB', 'javascript': '#F7DF1E', 'java': '#E76F00'},
    template='plotly_white',
)

# Anotacion del cruce Python > Java
if cross_py_java:
    fig_langs.add_annotation(
        x=cross_py_java['date'],
        y=cross_py_java['val_a'],
        text=f"Python supera a Java<br>({str(cross_py_java['date'])[:7]})",
        showarrow=True, arrowhead=2,
        ax=-80, ay=-40,
        font=dict(size=12, color='#3776AB'),
        bordercolor='#3776AB', borderwidth=1, borderpad=4,
        bgcolor='rgba(255,255,255,0.85)',
    )

fig_langs.update_layout(
    height=520,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    hovermode='x unified',
)
fig_langs.show()

### Análisis: Lenguajes y Mercado Laboral LATAM

El gráfico confirma un fenómeno estructural:

- **JavaScript** mantiene el liderazgo absoluto, coherente con su posición como tecnología universal de la web (frontend + backend vía Node.js). En mercados LATAM, la demanda de desarrolladores JS/TS sigue liderando los portales de empleo (Computrabajo, LinkedIn).
- **Python superó a Java** como segunda tecnología más consultada. Este cruce refleja la explosión de los ecosistemas de *Data Science*, *Machine Learning* e IA generativa, sectores con crecimiento acelerado en Ecuador y la región.
- **Java** muestra un declive sostenido en volumen de preguntas, pero mantiene relevancia en entornos corporativos (banca, telecomunicaciones). Su caída en Stack Overflow no implica desaparición, sino *madurez* — los problemas ya están resueltos y documentados.

**Implicación curricular para UNIANDES:** La enseñanza de Python como primer lenguaje y su integración transversal en asignaturas de IA, estadística y automatización es la apuesta correcta. Java debe mantenerse en el sílabo de Ingeniería de Software con énfasis en arquitectura empresarial (Spring Boot/microservicios).

---

## 2 · Guerras de Frameworks Web: React vs Angular vs Vue

El ecosistema frontend ha vivido una rotación acelerada de frameworks. Esta sección evalúa cuál de las tres opciones dominantes ha capturado la mayor cuota de atención técnica.

In [7]:
FW = ['reactjs', 'angular', 'vue.js']

df_fw = build_monthly_series(lf_base, FW)

# Detectar cruce React > Angular
cross_react_angular = find_crossover(df_fw, 'reactjs', 'angular')
if cross_react_angular:
    print(f"Cruce React > Angular detectado en: {cross_react_angular['date']}")
else:
    print('No se detecto cruce React > Angular en el periodo.')

Cruce React > Angular detectado en: 2019-02-01 00:00:00


In [8]:
fig_fw = px.line(
    df_fw.to_pandas(),
    x='YearMonth', y='MA3', color='Tag',
    title='Evolucion Mensual — React vs Angular vs Vue.js (MA 3 meses)',
    labels={'MA3': 'Preguntas (Media Movil 3m)', 'YearMonth': '', 'Tag': 'Framework'},
    color_discrete_map={'reactjs': '#61DAFB', 'angular': '#DD0031', 'vue.js': '#42B883'},
    template='plotly_white',
)

if cross_react_angular:
    fig_fw.add_annotation(
        x=cross_react_angular['date'],
        y=cross_react_angular['val_a'],
        text=f"React supera a Angular<br>({str(cross_react_angular['date'])[:7]})",
        showarrow=True, arrowhead=2,
        ax=-80, ay=-40,
        font=dict(size=12, color='#61DAFB'),
        bordercolor='#61DAFB', borderwidth=1, borderpad=4,
        bgcolor='rgba(255,255,255,0.85)',
    )

fig_fw.update_layout(
    height=520,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    hovermode='x unified',
)
fig_fw.show()

### Análisis: Implicación Curricular para Desarrollo Web en UNIANDES

Los datos revelan una dinámica clara:

- **React** domina de forma contundente. Su modelo de componentes, el ecosistema Next.js y la demanda de talento React en la región lo convierten en la apuesta más segura para el sílabo de la materia de **Desarrollo Web**.
- **Angular** muestra una caída progresiva. Si bien sigue siendo relevante en proyectos empresariales (entidades gubernamentales, ERP), su curva de aprendizaje y la fragmentación de versiones han reducido la adopción nueva.
- **Vue.js** permanece como un nicho estable. Su simplicidad lo hace ideal para prototipos rápidos y proyectos académicos, pero no alcanza la masa crítica profesional de React.

**Recomendación directa:** La materia de Desarrollo Web debe priorizar React (con TypeScript) como framework principal, manteniendo una unidad introductoria de Vue.js para ampliar perspectiva. Angular puede abordarse opcionalmente en electivas de Ingeniería de Software Empresarial.

---

## 3 · Paradigmas de Bases de Datos: PostgreSQL vs MongoDB vs MySQL

La década pasada vio un hype masivo por las bases de datos NoSQL. ¿Los datos de Stack Overflow confirman que el modelo relacional ha recuperado terreno? ¿O MongoDB consolidó su posición?

In [9]:
DB = ['postgresql', 'mongodb', 'mysql']

df_db = build_monthly_series(lf_base, DB)

# Detectar si PostgreSQL supero a MongoDB
cross_pg_mongo = find_crossover(df_db, 'postgresql', 'mongodb')
if cross_pg_mongo:
    print(f"Cruce PostgreSQL > MongoDB detectado en: {cross_pg_mongo['date']}")
else:
    print('No se detecto cruce PostgreSQL > MongoDB en el periodo.')

Cruce PostgreSQL > MongoDB detectado en: 2018-06-01 00:00:00


In [10]:
fig_db = px.line(
    df_db.to_pandas(),
    x='YearMonth', y='MA3', color='Tag',
    title='Evolucion Mensual — PostgreSQL vs MongoDB vs MySQL (MA 3 meses)',
    labels={'MA3': 'Preguntas (Media Movil 3m)', 'YearMonth': '', 'Tag': 'Base de Datos'},
    color_discrete_map={'postgresql': '#336791', 'mongodb': '#47A248', 'mysql': '#4479A1'},
    template='plotly_white',
)

if cross_pg_mongo:
    fig_db.add_annotation(
        x=cross_pg_mongo['date'],
        y=cross_pg_mongo['val_a'],
        text=f"PostgreSQL supera a MongoDB<br>({str(cross_pg_mongo['date'])[:7]})",
        showarrow=True, arrowhead=2,
        ax=-80, ay=-40,
        font=dict(size=12, color='#336791'),
        bordercolor='#336791', borderwidth=1, borderpad=4,
        bgcolor='rgba(255,255,255,0.85)',
    )

fig_db.update_layout(
    height=520,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    hovermode='x unified',
)
fig_db.show()

### Análisis: SQL vs NoSQL y el Sílabo de Bases de Datos

Los datos cuentan una historia interesante:

- **MySQL** fue históricamente el líder absoluto en consultas, pero muestra una **caída sostenida**. Su uso sigue siendo masivo (WordPress, aplicaciones LAMP legacy), pero los nuevos proyectos migran a alternativas.
- **PostgreSQL** ha experimentado un **crecimiento notable**, impulsado por su soporte de datos semiestructurados (JSONB), extensiones de ML (`pgvector`) y su adopción como base de datos estándar en startups y plataformas cloud (Supabase, Neon, Railway).
- **MongoDB** tuvo su pico durante el boom de las aplicaciones MEAN/MERN, pero su volumen de consultas se ha estabilizado o incluso retrocedido. El modelo "document-oriented" sigue siendo valioso, pero la flexibilidad de PostgreSQL ha capturado parte de su terreno.

**Implicación para el sílabo:**
1. **PostgreSQL** debe ser la base de datos relacional enseñada en primer lugar (reemplazando la tradición de MySQL).
2. **MongoDB** merece una unidad práctica en el módulo de NoSQL, pero contextualizada: enseñar *cuándo usarla* (catálogos, datos heterogéneos) en vez de presentarla como reemplazo universal de SQL.
3. El sílabo debe incluir un laboratorio comparativo SQL vs NoSQL sobre el mismo dataset para desarrollar criterio de selección en los estudiantes.

---

## 4 · Validación de Hipótesis

A continuación consolidamos los hallazgos de las tres series comparativas en una tabla formal que confronta las hipótesis iniciales con la evidencia empírica observada.

In [11]:
# Tabla de validacion de hipotesis
hipotesis_data = {
    'Hipotesis_Inicial': [
        'H1: Python supero a Java como segundo lenguaje mas consultado en Stack Overflow entre 2015-2024, impulsado por el ecosistema de IA/ML.',
        'H2: React se consolido como el framework frontend dominante, desplazando a Angular a partir de ~2018.',
        'H3: El modelo relacional (PostgreSQL) ha recuperado terreno frente a MongoDB en los ultimos anos, revirtiendo parcialmente el hype NoSQL.',
    ],
    'Resultado_Observado': [
        f"Cruce detectado: {str(cross_py_java['date'])[:7] if cross_py_java else 'No detectado'}. "
        'Python muestra una trayectoria ascendente sostenida mientras Java declina de forma consistente. '
        'JavaScript mantiene el liderazgo absoluto.',
        
        f"Cruce detectado: {str(cross_react_angular['date'])[:7] if cross_react_angular else 'No detectado'}. "
        'React muestra un crecimiento acelerado y domina ampliamente. '
        'Vue.js permanece como nicho estable con baja cuota relativa.',
        
        f"Cruce PG > MongoDB: {str(cross_pg_mongo['date'])[:7] if cross_pg_mongo else 'No detectado'}. "
        'MySQL lidera en volumen pero cae. PostgreSQL crece de forma sostenida. '
        'MongoDB se estabiliza o decrece.',
    ],
    'Veredicto': [
        'Confirmada' if cross_py_java else 'Parcialmente confirmada',
        'Confirmada' if cross_react_angular else 'Parcialmente confirmada',
        'Confirmada' if cross_pg_mongo else 'Parcialmente confirmada',
    ]
}

df_hipotesis = pl.DataFrame(hipotesis_data)

# Mostrar con estilo usando Pandas Styler
styled = (
    df_hipotesis.to_pandas()
    .style
    .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap', 'max-width': '400px'})
    .map(
        lambda v: 'background-color: #d4edda; font-weight: bold' if v == 'Confirmada'
        else ('background-color: #fff3cd; font-weight: bold' if 'Parcialmente' in str(v)
        else ('background-color: #f8d7da; font-weight: bold' if v == 'Refutada' else '')),
        subset=['Veredicto']
    )
    .set_caption('Tabla de Validacion de Hipotesis — Series Comparativas')
)

display(styled)

,Hipotesis_Inicial,Resultado_Observado,Veredicto
0,"H1: Python supero a Java como segundo lenguaje mas consultado en Stack Overflow entre 2015-2024, impulsado por el ecosistema de IA/ML.",Cruce detectado: 2017-06. Python muestra una trayectoria ascendente sostenida mientras Java declina de forma consistente. JavaScript mantiene el liderazgo absoluto.,Confirmada
1,"H2: React se consolido como el framework frontend dominante, desplazando a Angular a partir de ~2018.",Cruce detectado: 2019-02. React muestra un crecimiento acelerado y domina ampliamente. Vue.js permanece como nicho estable con baja cuota relativa.,Confirmada
2,"H3: El modelo relacional (PostgreSQL) ha recuperado terreno frente a MongoDB en los ultimos anos, revirtiendo parcialmente el hype NoSQL.",Cruce PG > MongoDB: 2018-06. MySQL lidera en volumen pero cae. PostgreSQL crece de forma sostenida. MongoDB se estabiliza o decrece.,Confirmada


---

## 5 · Exportación para el Dashboard (Fase 4)

Consolidamos las tres series mensuales en un único DataFrame maestro y lo exportamos como Parquet.
Este archivo será consumido directamente por:
- **Streamlit** para el gráfico interactivo de evolución temporal.
- **FastAPI** como endpoint de datos históricos.

In [12]:
# Unir las tres series en un DataFrame maestro
df_master = pl.concat([df_langs, df_fw, df_db])

# Agregar columna de categoria competitiva para facilitar filtros en el dashboard
df_master = df_master.with_columns(
    pl.when(pl.col('Tag').is_in(['python', 'javascript', 'java']))
      .then(pl.lit('Lenguajes'))
    .when(pl.col('Tag').is_in(['reactjs', 'angular', 'vue.js']))
      .then(pl.lit('Frameworks Web'))
    .when(pl.col('Tag').is_in(['postgresql', 'mongodb', 'mysql']))
      .then(pl.lit('Bases de Datos'))
    .otherwise(pl.lit('Otros'))
    .alias('Batalla')
)

# Exportar
os.makedirs(OUT_DIR, exist_ok=True)
output_path = os.path.join(OUT_DIR, 'series_mensuales.parquet')
df_master.write_parquet(output_path)

print(f'Serie maestro exportada: {output_path}')
print(f'  Filas: {df_master.shape[0]:,}')
print(f'  Columnas: {df_master.columns}')
print(f"  Tags unicos: {df_master['Tag'].unique().to_list()}")
display(df_master.head(10))

Serie maestro exportada: ../data/datos_procesados/eda\series_mensuales.parquet
  Filas: 999
  Columnas: ['Tag', 'YearMonth', 'Count', 'MA3', 'Batalla']
  Tags unicos: ['javascript', 'angular', 'vue.js', 'mysql', 'postgresql', 'java', 'python', 'reactjs', 'mongodb']


Tag,YearMonth,Count,MA3,Batalla
str,datetime[μs],u32,f64,str
"""java""",2015-01-01 00:00:00,16501,null,"""Lenguajes"""
"""java""",2015-02-01 00:00:00,16712,null,"""Lenguajes"""
"""java""",2015-03-01 00:00:00,19709,17640.666667,"""Lenguajes"""
"""java""",2015-04-01 00:00:00,19752,18724.333333,"""Lenguajes"""
"""java""",2015-05-01 00:00:00,18511,19324.0,"""Lenguajes"""
"""java""",2015-06-01 00:00:00,17829,18697.333333,"""Lenguajes"""
"""java""",2015-07-01 00:00:00,18668,18336.0,"""Lenguajes"""
"""java""",2015-08-01 00:00:00,17075,17857.333333,"""Lenguajes"""
"""java""",2015-09-01 00:00:00,16930,17557.666667,"""Lenguajes"""


---

> **Siguiente paso →** En la Fase 4 estos datos alimentarán el componente `st.line_chart` del dashboard Streamlit y el endpoint `/api/v1/series` de FastAPI, permitiendo al usuario explorar interactivamente la evolución de cualquier batalla tecnológica.